# Pipeline v3.0 — escenarios temporales de predicción (Google Colab o local)

Interfaz delgada sobre el paquete `eduoutcomes`. Evalúa por separado **T0-strict**, **T0** (ingreso), **T1** (fin del
1.er semestre) y **T2** (fin del 1.er año; = diseño v2.0) con el mismo protocolo y la misma reserva, y los compara.

* `configs/quick.yaml`: prueba estructural con datos reales (**no reportable**).
* `configs/default.yaml`: diseño preespecificado (reportable). Con 16 núcleos, ≈ 2 h por escenario.

**Interpretación obligatoria.** *Enrolled* es la situación al término de la duración prevista (no permanencia
longitudinal); los resultados son retrospectivos y solo T0/T0-strict corresponden al ingreso.

In [ ]:
# Colab: clona el repositorio e instala (en local, omite esta celda y usa `pip install -e ".[dev]"`)
import os, sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/DanielAndradeGiron/multiclass-student-outcomes.git
    %cd multiclass-student-outcomes
    !git checkout -q v3.0-temporal-scenarios || true
    !pip install -q -e .
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
from eduoutcomes.config import load_config
from eduoutcomes.scenarios import run_scenarios
from eduoutcomes.temporal import run_temporal

CONFIG = "configs/quick.yaml"          # cambia a configs/default.yaml para la corrida reportable
cfg = load_config(CONFIG, overrides=["output_dir='outputs/scenarios'"])
print(cfg.scenarios, "| quick =", cfg.quick)

In [ ]:
# 1) Los cuatro escenarios + comparación (Tablas 10–12, S12–S13, Fig. 8)
out = run_scenarios(cfg)
out

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
for t in ["Table10_scenarios_cv", "Table11_scenarios_holdout", "Table12_scenario_contrasts"]:
    display(Markdown(f"**{t}**")); display(pd.read_csv(out / "tables" / f"{t}.csv"))
display(Image(str(out / "figures" / "Fig8_prediction_scenarios.png"), width=900))
display(Markdown((out / "results_scenarios_draft.md").read_text(encoding="utf-8")))

In [ ]:
# 2) Exploratorio: validación temporal por cohorte inferida (solo conjunto de desarrollo)
tv = run_temporal(cfg)
pd.read_csv(tv / "supplementary" / "TableS14_temporal_validation.csv")

In [ ]:
# 3) Auditoría temporal de los predictores y cohortes (por escenario, p. ej. T2)
s = out.parent / "T2" / "supplementary"
display(pd.read_csv(s / "TableS1_data_audit.csv")); display(pd.read_csv(s / "TableS16_timing_uncertain_predictors.csv"))
display(pd.read_csv(s / "TableS15_inferred_cohorts.csv")); display(pd.read_csv(s / "TableS11_lime_sensitivity.csv"))